In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('AWCustomers.csv')

In [5]:
df.head()

,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,AddressLine1,AddressLine2,City,StateProvinceName,...,Education,Occupation,Gender,MaritalStatus,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,LastUpdated
0,21173,NaN,Chad,C,Yuan,NaN,7090 C. Mount Hood,NaN,Wollongong,New South Wales,...,Bachelors,Clerical,M,M,1,3,0,1,81916,2017-03-06
1,13249,NaN,Ryan,NaN,Perry,NaN,3651 Willow Lake Rd,NaN,Shawnee,British Columbia,...,Partial College,Clerical,M,M,1,2,1,2,81076,2017-03-06
2,29350,NaN,Julia,NaN,Thompson,NaN,1774 Tice Valley Blvd.,NaN,West Covina,California,...,Bachelors,Clerical,F,S,0,3,0,0,86387,2017-03-06
3,13503,NaN,Theodore,NaN,Gomez,NaN,2103 Baldwin Dr,NaN,Liverpool,England,...,Partial College,Skilled Manual,M,M,1,2,1,2,61481,2017-03-06
4,22803,NaN,Marshall,J,Shan,NaN,Am Gallberg 234,NaN,Werne,Nordrhein-Westfalen,...,Partial College,Skilled Manual,M,S,1,1,0,0,51804,2017-03-06


In [6]:
# comvert dob to age
df['BirthDate'] = pd.to_datetime(df['BirthDate'], errors='coerce')
df['Age'] = (pd.Timestamp.now().year - df['BirthDate'].dt.year)

In [7]:
selected_columns = [
    "CustomerID",
    'StateProvinceName',
    'CountryRegionName',
    'Age',
    'Education',
    'Occupation',
    'Gender',
    'MaritalStatus',
    'HomeOwnerFlag',
    'NumberCarsOwned',
    'NumberChildrenAtHome',
    'TotalChildren',
    'YearlyIncome'
]

In [8]:
df = df[selected_columns]

In [9]:
df.head()

,CustomerID,StateProvinceName,CountryRegionName,Age,Education,Occupation,Gender,MaritalStatus,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome
0,21173,New South Wales,Australia,38,Bachelors,Clerical,M,M,1,3,0,1,81916
1,13249,British Columbia,Canada,53,Partial College,Clerical,M,M,1,2,1,2,81076
2,29350,California,United States,40,Bachelors,Clerical,F,S,0,3,0,0,86387
3,13503,England,United Kingdom,48,Partial College,Skilled Manual,M,M,1,2,1,2,61481
4,22803,Nordrhein-Westfalen,Germany,50,Partial College,Skilled Manual,M,S,1,1,0,0,51804


In [10]:
# preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, KBinsDiscretizer

In [11]:
# (a) Handling Null values — fill with sensible defaults
# Numerical → median, Categorical → mode
for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

In [12]:
# Normalizing

In [13]:
from sklearn.preprocessing import MinMaxScaler

# Select continuous columns for normalization
continuous_cols = ['Age', 'YearlyIncome']

# Initialize Min-Max Scaler
scaler = MinMaxScaler()

# Fit and transform, then assign back to DataFrame
df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

print(df[continuous_cols].head())


        Age  YearlyIncome
0  0.185714      0.496842
1  0.400000      0.489453
2  0.214286      0.536172
3  0.328571      0.317083
4  0.357143      0.231958


In [14]:
# Discretization

In [15]:
from sklearn.preprocessing import KBinsDiscretizer

# -----------------------------
# 1. Discretization of Continuous Attributes
# -----------------------------

# Age → 3 bins (equal width)
age_bins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
df['Age_binned'] = age_bins.fit_transform(df[['Age']])

# YearlyIncome → 4 bins (equal frequency)
income_bins = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df['Income_binned'] = income_bins.fit_transform(df[['YearlyIncome']])

# -----------------------------
# 2. Binning High-Cardinality Categorical Attributes
# -----------------------------

# Keep top 10 most common Occupations, group rest as 'Other'
top_10_occupations = df['Occupation'].value_counts().index[:10]
df['Occupation_binned'] = df['Occupation'].where(
    df['Occupation'].isin(top_10_occupations),
    'Other'
)

# -----------------------------
# 3. Preview
# -----------------------------
print(df[['Age', 'Age_binned', 'YearlyIncome', 'Income_binned', 'Occupation', 'Occupation_binned']].head())


        Age  Age_binned  YearlyIncome  Income_binned      Occupation  \
0  0.185714         0.0      0.496842            2.0        Clerical   
1  0.400000         1.0      0.489453            2.0        Clerical   
2  0.214286         0.0      0.536172            2.0        Clerical   
3  0.328571         0.0      0.317083            1.0  Skilled Manual   
4  0.357143         1.0      0.231958            0.0  Skilled Manual   

  Occupation_binned  
0          Clerical  
1          Clerical  
2          Clerical  
3    Skilled Manual  
4    Skilled Manual  


/home/himanshu/assignments/machine_learning/assignment_02/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [16]:
# Normalization/Scaling

In [17]:
from sklearn.preprocessing import StandardScaler

# Select count-based numeric columns
count_cols = ['NumberCarsOwned', 'NumberChildrenAtHome', 'TotalChildren']

# Initialize standard scaler
scaler_std = StandardScaler()

# Fit and transform
df[count_cols] = scaler_std.fit_transform(df[count_cols])

print(df[count_cols].head())


   NumberCarsOwned  NumberChildrenAtHome  TotalChildren
0         1.892524             -0.594371       0.161342
1         0.798389              1.163279       1.239753
2         1.892524             -0.594371      -0.917069
3         0.798389              1.163279       1.239753
4        -0.295746             -0.594371      -0.917069


In [19]:
# One Hot Encoding
from sklearn.preprocessing import OneHotEncoder

# Categorical columns to encode
categorical_cols = [
    'StateProvinceName',
    'CountryRegionName',
    'Education',
    'Gender',
    'MaritalStatus',
    'HomeOwnerFlag',
    'Occupation_binned'  # from discretization step
]

# Initialize encoder (drop='first' to avoid dummy variable trap)
encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit and transform
encoded_array = encoder.fit_transform(df[categorical_cols])

# Create DataFrame for encoded columns
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=df.index
)

# Merge with original DataFrame
df_final = pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)

print(df_final.head())


   CustomerID       Age      Occupation  NumberCarsOwned  \
0       21173  0.185714        Clerical         1.892524   
1       13249  0.400000        Clerical         0.798389   
2       29350  0.214286        Clerical         1.892524   
3       13503  0.328571  Skilled Manual         0.798389   
4       22803  0.357143  Skilled Manual        -0.295746   

   NumberChildrenAtHome  TotalChildren  YearlyIncome  Age_binned  \
0             -0.594371       0.161342      0.496842         0.0   
1              1.163279       1.239753      0.489453         1.0   
2             -0.594371      -0.917069      0.536172         0.0   
3              1.163279       1.239753      0.317083         0.0   
4             -0.594371      -0.917069      0.231958         1.0   

   Income_binned  StateProvinceName_Alberta  ...  Education_High School  \
0            2.0                        0.0  ...                    0.0   
1            2.0                        0.0  ...                    0.0   
2    

In [20]:
# Part 3

In [21]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Assume df_final is your fully transformed dataset
# Select two objects (rows) for similarity
obj1 = df_final.iloc[0]
obj2 = df_final.iloc[1]

# ------------------------------
# (a) Similarity Calculations
# ------------------------------

# --- Simple Matching Coefficient (SMC) for binary attributes ---
def simple_matching_coefficient(x, y):
    matches = np.sum(x == y)
    return matches / len(x)

# --- Jaccard Similarity for binary attributes ---
def jaccard_similarity(x, y):
    intersection = np.sum((x == 1) & (y == 1))
    union = np.sum((x == 1) | (y == 1))
    return intersection / union if union != 0 else 0

# --- Cosine Similarity for numeric attributes ---
def cosine_sim(x, y):
    return cosine_similarity([x], [y])[0][0]

# Example: separate binary vs numeric attributes
binary_cols = [col for col in df_final.columns if set(df_final[col].unique()) <= {0, 1}]
numeric_cols = [col for col in df_final.columns if col not in binary_cols]

# Compute similarities
smc_value = simple_matching_coefficient(obj1[binary_cols], obj2[binary_cols])
jaccard_value = jaccard_similarity(obj1[binary_cols], obj2[binary_cols])
cosine_value = cosine_sim(obj1[numeric_cols], obj2[numeric_cols])

print("Simple Matching Coefficient (Binary):", smc_value)
print("Jaccard Similarity (Binary):", jaccard_value)
print("Cosine Similarity (Numeric):", cosine_value)


ValueError: could not convert string to float: 'Clerical'